# 14 — Uncertainty: variance split + reliability

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-systems-today/cubespec/blob/main/notebooks/14_uncertainty_reliability.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ai-systems-today/cubespec/main?urlpath=lab/tree/notebooks/14_uncertainty_reliability.ipynb) [![nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/ai-systems-today/cubespec/blob/main/notebooks/14_uncertainty_reliability.ipynb) [![Dashboard](https://img.shields.io/badge/open-dashboard-2563eb)](https://sensitive-spark.lovable.app)

**Abstract.** Decompose variance into aleatory and epistemic components, compute the reliability index β and the exceedance curve.

**Mirrors.** **Report** tab → *Uncertainty* panel (variance split + reliability + threshold input + Wilson CI).


In [ ]:
# cubespec-bootstrap
# Install cubespec on first run (Colab/Binder safe; no-op if already importable).
try:
    import cubespec  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/ai-systems-today/cubespec.git'])
    import cubespec  # noqa: F401


## 1. Variance decomposition

For each output we split the variance into:

* **Aleatory**  — irreducible scatter from the input distributions.
* **Epistemic** — surrogate uncertainty (calibration residuals).
* **Total**     — sum, used to size confidence bands.


In [ ]:
from cubespec import (
    DEFAULT_CSP, sample_independent, compute_outputs_batch, decompose_variance,
)
import numpy as np, pandas as pd

Y = compute_outputs_batch(sample_independent(DEFAULT_CSP, n=20_000, seed=1337))
split = decompose_variance(Y)
pd.DataFrame([{'output': k, 'aleatory': v.aleatory,
               'epistemic': v.epistemic, 'total': v.total}
              for k, v in split.items()])


## 2. Reliability index β and exceedance probability P(P9 ≥ τ)

We compute β and P(P9 ≥ τ) for three thresholds: μ − 1σ, μ − 2σ, μ − 3σ. These mirror the editable threshold input in the Uncertainty panel.


In [ ]:
from cubespec import reliability_index
P9 = Y[:, 2]
rows = []
for k in (1, 2, 3):
    tau = float(P9.mean() - k * P9.std(ddof=1))
    r = reliability_index(Y, threshold=tau, output='P9_compressive_strength', direction='ge')
    rows.append({'tau (MPa)': round(tau, 3), 'beta': r.beta, 'P(P9>=tau)': r.p,
                 'wilson_lo': r.p_lo, 'wilson_hi': r.p_hi})
pd.DataFrame(rows)


## 3. Exceedance curve (sweep τ)


In [ ]:
import matplotlib.pyplot as plt
taus = np.linspace(P9.min(), P9.max(), 60)
probs = np.array([(P9 >= t).mean() for t in taus])
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(taus, probs, color='#5b8def', lw=2)
ax.axhline(0.95, color='#e94f64', ls='--', lw=0.8); ax.text(taus[1], 0.96, '95% target', color='#e94f64', fontsize=9)
ax.set_xlabel('threshold τ (MPa)'); ax.set_ylabel('P(P9 ≥ τ)')
ax.set_title('Exceedance curve for P9'); fig.tight_layout(); plt.show()


## 4. Plain-language summary

* Aleatory variance dominates → reduce input scatter (tighter QC) to improve P9.
* Epistemic dominates → invest in surrogate retraining.
* β > 3 at your target τ → the design is comfortably above the characteristic strength.


In [ ]:
# cubespec-reproducibility-footer
import platform, sys, numpy as np, scipy, pandas as pd, cubespec
print('Reproducibility')
print('  cubespec :', cubespec.__version__)
print('  python   :', sys.version.split()[0], 'on', platform.system())
print('  numpy    :', np.__version__)
print('  scipy    :', scipy.__version__)
print('  pandas   :', pd.__version__)
